In [ ]:
!pip -q install feedparser requests pandas gspread google-auth python-dateutil

In [ ]:
import requests
import feedparser
import pandas as pd

from urllib.parse import quote
from dateutil import parser

from google.colab import auth
import google.auth
import gspread

print("Authenticating Google Account...")

auth.authenticate_user()

credentials, _ = google.auth.default()

gc = gspread.authorize(credentials)

print("Google Authentication Successful!")

Authenticating Google Account...
Google Authentication Successful!


In [ ]:
# Get your free API key from:
# https://gnews.io/
GNEWS_API_KEY = ""

# Get your free API key from:
# https://newsdata.io/
NEWSDATA_API_KEY = ""


GOOGLE_SHEET_URL = ""


WORKSHEET_NAME = "Sheet1"



SEARCH_KEYWORDS = [

    "NEET paper leak",

    "NEET protest",

    "CJP protest",

    "NEET CJP",

    "NEET scam",

    "NTA protest",

    "NEET irregularities"

]

EXAM_DATABASE = {

    "NEET": {
        "Name of Exam": "NEET UG",
        "Exam Category": "Medical Entrance",
        "Board": "NTA",
        "Conducted By": "NTA",
        "PBT/CBT": "PBT"
    }

}

CATEGORY_KEYWORDS = {

    "Paper Leak": [
        "paper leak",
        "question paper leak",
        "leaked paper",
        "question leak"
    ],

    "Protest": [
        "protest",
        "students protest",
        "demonstration",
        "agitation"
    ],

    "Court Case": [
        "court",
        "hearing",
        "judge",
        "bail"
    ],

    "Investigation": [
        "cbi",
        "investigation",
        "probe"
    ],

    "Arrest": [
        "arrest",
        "arrested"
    ],

    "Malpractice": [
        "cheating",
        "fraud",
        "scam",
        "irregularities"
    ]

}

INDIAN_STATES = [

"Delhi",
"Punjab",
"Haryana",
"Rajasthan",
"Bihar",
"Jharkhand",
"Uttar Pradesh",
"Madhya Pradesh",
"Maharashtra",
"Gujarat",
"West Bengal",
"Odisha",
"Assam",
"Telangana",
"Andhra Pradesh",
"Tamil Nadu",
"Karnataka",
"Kerala"

]


RSS_LANGUAGE = "en-IN"
RSS_COUNTRY = "IN"

MAX_RESULTS = 10

REQUEST_DELAY = 1


print("="*50)
print("NEET News Tracker Configuration")
print("="*50)

print(f"Keywords Loaded : {len(SEARCH_KEYWORDS)}")
print(f"GNews API       : {'Configured' if GNEWS_API_KEY != 'YOUR_GNEWS_API_KEY' else 'Not Configured'}")
print(f"NewsData API    : {'Configured' if NEWSDATA_API_KEY != 'YOUR_NEWSDATA_API_KEY' else 'Not Configured'}")
print(f"Worksheet       : {WORKSHEET_NAME}")

print("="*50)

NEET News Tracker Configuration
Keywords Loaded : 7
GNews API       : Configured
NewsData API    : Configured
Worksheet       : Sheet1


In [ ]:
import time
from urllib.parse import quote

def fetch_google_news():

    print("\nFetching news from Google News RSS...\n")

    articles = []

    for keyword in SEARCH_KEYWORDS:

        print(f"Searching: {keyword}")

        # Build Google News RSS URL
        rss_url = (
            "https://news.google.com/rss/search?"
            f"q={quote(keyword)}"
            "&hl=en-IN"
            "&gl=IN"
            "&ceid=IN:en"
        )

        feed = feedparser.parse(rss_url)

        print(f"Found {len(feed.entries)} articles")

        for item in feed.entries:

            articles.append({
                              "Title": item.title,
                              "Description": item.get("summary", ""),
                              "Source": "Google News RSS",
                              "Published Date": item.get("published", ""),
                              "URL": item.link,
                              "Keyword Matched": keyword
        })

        # Small delay to avoid sending requests too quickly
        time.sleep(REQUEST_DELAY)

    print("\nGoogle News RSS Finished.")
    print(f"Total Articles Collected: {len(articles)}\n")

    return articles

In [ ]:
rss_news = fetch_google_news()

print("First 5 Articles\n")

for article in rss_news[:5]:
    print(article)


Fetching news from Google News RSS...

Searching: NEET paper leak
Found 103 articles
Searching: NEET protest
Found 106 articles
Searching: CJP protest
Found 100 articles
Searching: NEET CJP
Found 100 articles
Searching: NEET scam
Found 100 articles
Searching: NTA protest
Found 99 articles
Searching: NEET irregularities
Found 100 articles

Google News RSS Finished.
Total Articles Collected: 708

First 5 Articles

{'Title': "Students Saying '1st It Hurt Me, Then I Took Advantage': Nirmala Sitharaman On Paper Leak - NDTV", 'Description': '<a href="https://news.google.com/rss/articles/CBMiwwFBVV95cUxPaDE1M0hacEhsRHN2SjJzTGN1WjloZDBxZENWYTdDRjBBZXM5NDRTeV9EdDJGZGpMMEgzaHZjblB3aHZjdW9aRjRNcGRxUHFZRUJmSm5KYndfeF9FVzNYOWFJc1RaZzZZb2V6cTBmRGhvZXRCUlVyYnE3dXFnUjNaNzZhWG9YNzdJdTdjVWJNNFh6dkZJMEszNldrOXNEUVVvSVczbHQ4cDFqWkMwSElLZnpCQ0J5T2RoTmZ2N3UwVmFCSnfSAcsBQVVfeXFMTTVBVUlkX21OdXhDYTFSbVJYRnUtc0g5U0NzUVpvZ0NLZDZFdW1LaE9QY0NQXzkxLU5VbjR4OHNwbTBQaGdZcFkxUjB6d243Rk5US1BHREI1YmJVWkNnTlFWZDUwelRsTWx

In [ ]:
import time

def fetch_gnews():

    print("\nFetching news from GNews API...\n")

    articles = []

    if GNEWS_API_KEY == "YOUR_GNEWS_API_KEY":
        print("GNews API Key not configured.")
        return articles

    for keyword in SEARCH_KEYWORDS:

        print(f"Searching: {keyword}")

        url = "https://gnews.io/api/v4/search"

        params = {
            "q": keyword,
            "lang": "en",
            "country": "in",
            "max": MAX_RESULTS,
            "token": GNEWS_API_KEY
        }

        try:

            response = requests.get(url, params=params, timeout=15)

            if response.status_code == 200:

                data = response.json()

                print(f"Found {len(data.get('articles', []))} articles")

                for item in data.get("articles", []):

                    articles.append({
                                    "Title": item.get("title", ""),
                                    "Description": item.get("description", ""),
                                    "Source": "GNews API",
                                    "Published Date": item.get("publishedAt", ""),
                                    "URL": item.get("url", ""),
                                    "Keyword Matched": keyword
            })

            else:

                print(f"Error {response.status_code}")

        except Exception as e:

            print("Request Failed:", e)

        time.sleep(REQUEST_DELAY)

    print("\nGNews API Finished.")
    print(f"Total Articles Collected: {len(articles)}\n")
    return articles

In [ ]:
gnews_articles = fetch_gnews()
print("First 5 Articles\n")
for article in gnews_articles[:5]:
    print(article)


Fetching news from GNews API...

Searching: NEET paper leak
Found 10 articles
Searching: NEET protest
Found 10 articles
Searching: CJP protest
Found 10 articles
Searching: NEET CJP
Found 10 articles
Searching: NEET scam
Found 10 articles
Searching: NTA protest
Found 10 articles
Searching: NEET irregularities
Found 10 articles

GNews API Finished.
Total Articles Collected: 70

First 5 Articles

{'Title': "Pralhad Joshi Takes Charge As Education Minister Amid India's Deepening Education Crisis", 'Description': 'Pralhad Joshi has taken additional charge as Union Education Minister after Dharmendra Pradhan resigned over the NEET-UG paper leak. He inherits an education system hit by paper leaks, CBSE evaluation controversies and growing youth anger. Experts say restoring trust through transparent, secure and accountable examinations will be his biggest challenge.', 'Source': 'GNews API', 'Published Date': '2026-07-26T18:13:17Z', 'URL': 'https://www.freepressjournal.in/education/pralhad-jos

In [ ]:
import time

def fetch_newsdata():

    print("\nFetching news from NewsData.io...\n")

    articles = []

    if NEWSDATA_API_KEY == "YOUR_NEWSDATA_API_KEY":
        print("NewsData API Key not configured.")
        return articles

    for keyword in SEARCH_KEYWORDS:

        print(f"Searching: {keyword}")

        url = "https://newsdata.io/api/1/news"

        params = {
            "apikey": NEWSDATA_API_KEY,
            "q": keyword,
            "country": "in",
            "language": "en"
        }

        try:

            response = requests.get(url, params=params, timeout=15)

            if response.status_code == 200:

                data = response.json()

                news = data.get("results", [])

                print(f"Found {len(news)} articles")

                for item in news:

                    articles.append({
                                    "Title": item.get("title", ""),
                                    "Source": "NewsData.io",
                                    "Published Date": item.get("pubDate", ""),
                                    "URL": item.get("link", ""),
                                    "Keyword Matched": keyword
            })

            else:

                print(f"Error {response.status_code}")

                try:
                    print(response.json())
                except:
                    print(response.text)

        except Exception as e:

            print("Request Failed:", e)

        time.sleep(REQUEST_DELAY)

    print("\nNewsData.io Finished.")
    print(f"Total Articles Collected: {len(articles)}\n")

    return articles

In [ ]:
newsdata_articles = fetch_newsdata()
print("First 5 Articles\n")
for article in newsdata_articles[:5]:
    print(article)


Fetching news from NewsData.io...

Searching: NEET paper leak
Found 10 articles
Searching: NEET protest
Found 10 articles
Searching: CJP protest
Found 10 articles
Searching: NEET CJP
Found 10 articles
Searching: NEET scam
Found 10 articles
Searching: NTA protest
Found 10 articles
Searching: NEET irregularities
Found 10 articles

NewsData.io Finished.
Total Articles Collected: 70

First 5 Articles

{'Title': 'Dharmendra Pradhan’s exit not enough, punish officials behind police action: Congress', 'Source': 'NewsData.io', 'Published Date': '2026-07-26 18:13:00', 'URL': 'https://timesofindia.indiatimes.com/city/chandigarh/dharmendra-pradhans-exit-not-enough-punish-officials-behind-police-action-congress/articleshow/132646399.cms', 'Keyword Matched': 'NEET paper leak'}
{'Title': "Fast-Track Courts, Tougher Jail Terms: What Centre's New Anti-Paper Leak Bill Proposes", 'Source': 'NewsData.io', 'Published Date': '2026-07-26 18:11:28', 'URL': 'https://news.abplive.com/news/india/fast-track-cou

In [ ]:
import re

print("\nProcessing News...\n")


all_articles = rss_news + gnews_articles + newsdata_articles

print("Total Articles Collected :", len(all_articles))


unique_articles = []
seen_urls = set()

for article in all_articles:

    url = article.get("URL","").strip()

    if url == "":
        continue

    if url not in seen_urls:
        seen_urls.add(url)
        unique_articles.append(article)

print("Unique Articles :", len(unique_articles))


def detect_exam(text):

    text = text.upper()

    for exam in EXAM_DATABASE:

        if exam.upper() in text:

            return EXAM_DATABASE[exam]

    return {
        "Name of Exam":"Unknown",
        "Exam Category":"Unknown",
        "Board":"Unknown",
        "Conducted By":"Unknown",
        "PBT/CBT":"Unknown"
    }


def detect_category(text):

    text = text.lower()

    for category, words in CATEGORY_KEYWORDS.items():

        for word in words:

            if word.lower() in text:

                return category

    return "Other"


def detect_state(text):

    for state in INDIAN_STATES:

        if state.lower() in text.lower():

            return state

    return "Unknown"


def extract_reason(description):

    if description is None:
        return ""

    description = re.sub("<.*?>","",description)
    description = description.replace("\n"," ").strip()

    sentences = re.split(r'(?<=[.!?]) +', description)

    if len(sentences):

        return sentences[0][:250]

    return description[:250]


def extract_year(title, published_date):

    years = re.findall(r'20\d{2}', title)

    if years:

        return years[0]

    try:

        return str(pd.to_datetime(published_date).year)

    except:

        return ""

structured_news = []

for article in unique_articles:

    title = article.get("Title","")
    description = article.get("Description","")

    full_text = title + " " + description

    exam = detect_exam(full_text)

    state = detect_state(full_text)

    structured_news.append({

        "Name of Exam": exam["Name of Exam"],

        "Exam Category": exam["Exam Category"],

        "Board": exam["Board"],

        "State": state,

        "Conducted In": "India" if state=="Unknown" else state,

        "Conducted By": exam["Conducted By"],

        "Link": article["URL"],

        "Category": detect_category(full_text),

        "Reason": extract_reason(description),

        "PBT/CBT": exam["PBT/CBT"],

        "Exam Year": extract_year(title, article["Published Date"]),

        "Published Date": article["Published Date"]

    })

news_df = pd.DataFrame(structured_news)

news_df["Published Date"] = pd.to_datetime(
    news_df["Published Date"],
    errors="coerce"
)

news_df = news_df.sort_values(
    by="Published Date",
    ascending=False
)

news_df.reset_index(drop=True, inplace=True)

print("\nStructured Dataset Created Successfully!\n")

print("Total Rows :", len(news_df))

display(news_df.head(20))


Processing News...

Total Articles Collected : 848
Unique Articles : 750

Structured Dataset Created Successfully!

Total Rows : 750


,Name of Exam,Exam Category,Board,State,Conducted In,Conducted By,Link,Category,Reason,PBT/CBT,Exam Year,Published Date
0,NEET UG,Medical Entrance,NTA,Unknown,India,NTA,https://news.google.com/rss/articles/CBMi0AFBV...,Protest,NEET Row: Right to Peaceful Protest Guaranteed...,PBT,2026,2026-07-27 06:07:04
1,NEET UG,Medical Entrance,NTA,Unknown,India,NTA,https://news.google.com/rss/articles/CBMi7gFBV...,Protest,NEET protests: SC says right to peaceful prote...,PBT,2026,2026-07-27 06:06:24
2,NEET UG,Medical Entrance,NTA,Unknown,India,NTA,https://news.google.com/rss/articles/CBMi1gFBV...,Protest,Congress MPs protest alleged 'police brutality...,PBT,2026,2026-07-27 06:05:48
3,NEET UG,Medical Entrance,NTA,Unknown,India,NTA,https://news.google.com/rss/articles/CBMivAFBV...,Protest,WB CM Alleges Fundamentalist Links to Kolkata ...,PBT,2026,2026-07-27 06:05:19
4,NEET UG,Medical Entrance,NTA,Unknown,India,NTA,https://news.google.com/rss/articles/CBMi-AFBV...,Protest,CJI Surya Kant on police ‘brutality’ against N...,PBT,2026,2026-07-27 06:01:40
5,Unknown,Unknown,Unknown,Delhi,Delhi,Unknown,https://news.google.com/rss/articles/CBMi3AFBV...,Protest,'Mere agitation cannot justify lathi-charge': ...,Unknown,2026,2026-07-27 06:01:07
6,NEET UG,Medical Entrance,NTA,Unknown,India,NTA,https://news.google.com/rss/articles/CBMijAJBV...,Protest,NEET protest: SC says police excesses cannot b...,PBT,2026,2026-07-27 05:59:52
7,NEET UG,Medical Entrance,NTA,Unknown,India,NTA,https://news.google.com/rss/articles/CBMivAFBV...,Protest,NEET Row: Right To Peaceful Protest Guaranteed...,PBT,2026,2026-07-27 05:56:14
8,NEET UG,Medical Entrance,NTA,Unknown,India,NTA,https://news.google.com/rss/articles/CBMi8AFBV...,Protest,Opposition seeks Rajya Sabha discussion on all...,PBT,2026,2026-07-27 05:50:08
9,NEET UG,Medical Entrance,NTA,Unknown,India,NTA,https://news.google.com/rss/articles/CBMiugFBV...,Paper Leak,NEET Paper Leak: PM Modi Forms Exam Reform Tas...,PBT,2026,2026-07-27 05:46:05


In [ ]:
import re

print("\nConnecting to Google Sheet...\n")

spreadsheet_id = re.search(
    r"/spreadsheets/d/([a-zA-Z0-9-_]+)",
    GOOGLE_SHEET_URL
).group(1)

spreadsheet = gc.open_by_key(spreadsheet_id)

worksheet = spreadsheet.worksheet(WORKSHEET_NAME)

print("Google Sheet Connected Successfully!")


header = [

    "Name of Exam",
    "Exam Category",
    "Board",
    "State",
    "Conducted In",
    "Conducted By",
    "Link",
    "Category",
    "Reason",
    "PBT/CBT",
    "Exam Year",
    "Published Date"

]


if worksheet.get_all_values() == []:

    worksheet.append_row(header)

    print("Header Added")


rows = worksheet.get_all_values()

existing_links = set()

for row in rows[1:]:

    if len(row) >= 7:

        existing_links.add(row[6])

print("Existing Articles :", len(existing_links))


rows_to_upload = []

for _, row in news_df.iterrows():

    if row["Link"] not in existing_links:

        rows_to_upload.append([

            row["Name of Exam"],
            row["Exam Category"],
            row["Board"],
            row["State"],
            row["Conducted In"],
            row["Conducted By"],
            row["Link"],
            row["Category"],
            row["Reason"],
            row["PBT/CBT"],
            row["Exam Year"],
            str(row["Published Date"])

        ])

print("New Articles :", len(rows_to_upload))

if rows_to_upload:

    worksheet.append_rows(
        rows_to_upload,
        value_input_option="USER_ENTERED"
    )

    print(f"\n✅ Uploaded {len(rows_to_upload)} new records.")

else:

    print("\n✅ No New Articles Found.")


Connecting to Google Sheet...

Google Sheet Connected Successfully!
Existing Articles : 0
New Articles : 750

✅ Uploaded 750 new records.


In [ ]:
news_df.to_csv(
    "neet_news.csv",
    index=False
)
print("CSV Exported Successfully!")

CSV Exported Successfully!
